# GOLDS Model Zoo Cards

This notebook generates **model cards** for every trained agent in the GOLDS project.
A model card is a standardized summary that describes:

- What game the agent plays
- How it was trained (algorithm, hyperparameters)
- How well it performs (evaluation stats, comparison to baselines)

Model cards make it easy to compare agents and share reproducible results.

**Prerequisites:** Run at least one training job (e.g., `uv run golds train run --config configs/games/breakout.yaml`) so that `results.json` exists.

In [ ]:
# Standard imports used throughout the notebook
from __future__ import annotations

import json
import sys
from datetime import datetime
from pathlib import Path
from typing import Optional

# Make sure the GOLDS package is importable
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Optional: pandas for nicer tables (falls back to plain print)
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas not installed -- tables will use plain text formatting.")

print(f"Project root: {PROJECT_ROOT}")
print(f"pandas available: {HAS_PANDAS}")

---
## 1. Model Card Template

Every model card follows a consistent Markdown template.  This makes it easy to
scan across games and spot trends (e.g., which games are solved, which still
need more training).

Here is the template we will fill in programmatically later:

In [ ]:
MODEL_CARD_TEMPLATE = """\
# Model Card: {game_id}

## Overview
| Field               | Value              |
|---------------------|--------------------|
| **Game**            | {game_id}          |
| **Experiment**      | {experiment_name}  |
| **Algorithm**       | PPO                |
| **Reward regime**   | {reward_regime}    |
| **Device**          | {device}           |
| **Parallel envs**   | {n_envs}           |

## Training
| Metric                    | Value              |
|---------------------------|--------------------|
| **Target timesteps**      | {total_timesteps_target:,} |
| **Completed timesteps**   | {total_timesteps_completed:,} |
| **Wall-clock time**       | {wall_time}        |
| **Training round**        | {round}            |

## Evaluation
| Metric                    | Value              |
|---------------------------|--------------------|
| **Best eval reward**      | {best_eval_reward} |
| **Final eval reward**     | {final_eval_reward}|
| **Human score**           | {human_score}      |
| **Human-normalized**      | {human_normalized_score} |
| **Published PPO**         | {published_ppo_score} |

## Artifacts
- Best model: `{best_model_path}`
- Final model: `{final_model_path}`
- TensorBoard: `{tensorboard_log_dir}`

---
*Generated {generated_at}*
"""

print("Template loaded. Fields will be filled by the auto-generator in Section 4.")

---
## 2. Published Baselines

GOLDS ships with published baseline scores from the RL literature
(Mnih et al. 2015, Schulman et al. 2017, Machado et al. 2018).

These let us compare our agents to **human**, **random**, **DQN**, and **PPO**
reference points.

In [ ]:
from golds.results.baselines import BASELINES, BaselineScores

# Build a table of all published baselines
rows = []
for game_id, scores in sorted(BASELINES.items()):
    rows.append({
        "Game": game_id,
        "Human": scores.human,
        "Random": scores.random,
        "DQN": scores.dqn if scores.dqn is not None else "--",
        "PPO": scores.ppo if scores.ppo is not None else "--",
    })

if HAS_PANDAS:
    df_baselines = pd.DataFrame(rows)
    display(df_baselines.style.set_caption("Published Baseline Scores"))
else:
    # Plain text fallback
    header = f"{'Game':<25} {'Human':>10} {'Random':>10} {'DQN':>10} {'PPO':>10}"
    print(header)
    print("-" * len(header))
    for r in rows:
        print(f"{r['Game']:<25} {str(r['Human']):>10} {str(r['Random']):>10} {str(r['DQN']):>10} {str(r['PPO']):>10}")

---
## 3. Results Store

The `ResultStore` class persists every training run to `results.json`.
Let's load it and display what we have so far.

If you haven't trained anything yet, the cell below will show a helpful
message instead of crashing.

In [ ]:
from golds.results.store import ResultStore
from golds.results.schema import TrainingResult

RESULTS_PATH = PROJECT_ROOT / "results.json"

try:
    store = ResultStore(path=RESULTS_PATH)
    all_results = store.get_results()

    if not all_results:
        print("ResultStore is empty. Train a model first:")
        print("  uv run golds train run --config configs/games/breakout.yaml")
    else:
        print(f"Found {len(all_results)} training result(s) in {RESULTS_PATH}\n")

        rows = []
        for r in all_results:
            rows.append({
                "Game": r.game_id,
                "Experiment": r.experiment_name,
                "Round": r.round,
                "Timesteps": f"{r.total_timesteps_completed:,}",
                "Best Reward": f"{r.best_eval_reward:.1f}" if r.best_eval_reward is not None else "--",
                "Final Reward": f"{r.final_eval_reward:.1f}" if r.final_eval_reward is not None else "--",
                "HNS": f"{r.human_normalized_score:.2%}" if r.human_normalized_score is not None else "--",
                "Device": r.device,
            })

        if HAS_PANDAS:
            df_results = pd.DataFrame(rows)
            display(df_results.style.set_caption("All Training Results"))
        else:
            header = f"{'Game':<20} {'Experiment':<20} {'Rd':>3} {'Timesteps':>12} {'Best':>10} {'Final':>10} {'HNS':>8}"
            print(header)
            print("-" * len(header))
            for r in rows:
                print(f"{r['Game']:<20} {r['Experiment']:<20} {r['Round']:>3} {r['Timesteps']:>12} {r['Best Reward']:>10} {r['Final Reward']:>10} {r['HNS']:>8}")

except FileNotFoundError:
    print(f"No results file found at {RESULTS_PATH}.")
    print("Train a model first:")
    print("  uv run golds train run --config configs/games/breakout.yaml")
except json.JSONDecodeError as e:
    print(f"Error reading {RESULTS_PATH}: {e}")
except Exception as e:
    print(f"Could not load results: {e}")

---
## 4. Auto-Generating Model Cards

This function takes a `game_id`, pulls the best result from the `ResultStore`,
merges in published baselines, and renders a complete model card using the
template from Section 1.

In [ ]:
from IPython.display import Markdown, display as ipy_display
from golds.results.baselines import BASELINES, human_normalized_score


def format_wall_time(seconds: float) -> str:
    """Convert seconds to a human-readable string like '2h 15m 30s'."""
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    parts = []
    if h > 0:
        parts.append(f"{h}h")
    if m > 0:
        parts.append(f"{m}m")
    parts.append(f"{s}s")
    return " ".join(parts)


def _fmt(value, fmt=".1f", fallback="--"):
    """Format a numeric value, returning fallback for None."""
    if value is None:
        return fallback
    return f"{value:{fmt}}"


def generate_model_card(game_id: str, store: ResultStore) -> str:
    """Generate a Markdown model card for the best result of a given game.

    Parameters
    ----------
    game_id : str
        The game identifier, e.g. 'breakout', 'pong'.
    store : ResultStore
        A loaded ResultStore instance.

    Returns
    -------
    str
        Rendered Markdown model card.
    """
    result = store.get_best(game_id)
    if result is None:
        return f"# Model Card: {game_id}\n\nNo training results found for this game."

    # Compute human-normalized score from baselines if not already stored
    hns = result.human_normalized_score
    if hns is None and result.best_eval_reward is not None:
        hns = human_normalized_score(game_id, result.best_eval_reward)

    # Pull published scores
    baseline = BASELINES.get(game_id)
    human_sc = baseline.human if baseline else result.human_score
    ppo_sc = baseline.ppo if baseline else result.published_ppo_score

    card = MODEL_CARD_TEMPLATE.format(
        game_id=game_id,
        experiment_name=result.experiment_name,
        reward_regime=result.reward_regime,
        device=result.device,
        n_envs=result.n_envs,
        total_timesteps_target=result.total_timesteps_target,
        total_timesteps_completed=result.total_timesteps_completed,
        wall_time=format_wall_time(result.wall_time_seconds),
        round=result.round,
        best_eval_reward=_fmt(result.best_eval_reward),
        final_eval_reward=_fmt(result.final_eval_reward),
        human_score=_fmt(human_sc),
        human_normalized_score=f"{hns:.2%}" if hns is not None else "--",
        published_ppo_score=_fmt(ppo_sc),
        best_model_path=result.best_model_path or "--",
        final_model_path=result.final_model_path or "--",
        tensorboard_log_dir=result.tensorboard_log_dir or "--",
        generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
    )
    return card


print("generate_model_card() is ready. See examples below.")

In [ ]:
# Generate and display a model card for every game in the results store.
# If the store is empty, we show a demo card with placeholder values.

try:
    store = ResultStore(path=RESULTS_PATH)
    all_results = store.get_results()

    if all_results:
        # Get unique game IDs
        game_ids = sorted({r.game_id for r in all_results})
        print(f"Generating model cards for {len(game_ids)} game(s)...\n")

        for gid in game_ids:
            card_md = generate_model_card(gid, store)
            ipy_display(Markdown(card_md))
    else:
        print("No results in store. Showing a demo card with placeholder data:\n")
        demo_card = MODEL_CARD_TEMPLATE.format(
            game_id="breakout",
            experiment_name="demo_run",
            reward_regime="clipped",
            device="cpu",
            n_envs=16,
            total_timesteps_target=10_000_000,
            total_timesteps_completed=10_000_000,
            wall_time="2h 15m 30s",
            round=1,
            best_eval_reward="274.8",
            final_eval_reward="250.3",
            human_score="30.5",
            human_normalized_score="94.87%",
            published_ppo_score="274.8",
            best_model_path="outputs/breakout/best_model.zip",
            final_model_path="outputs/breakout/final_model.zip",
            tensorboard_log_dir="outputs/breakout/tb_logs",
            generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
        )
        ipy_display(Markdown(demo_card))

except Exception as e:
    print(f"Could not generate cards: {e}")
    print("Showing demo card instead:\n")
    demo_card = MODEL_CARD_TEMPLATE.format(
        game_id="breakout",
        experiment_name="demo_run",
        reward_regime="clipped",
        device="cpu",
        n_envs=16,
        total_timesteps_target=10_000_000,
        total_timesteps_completed=10_000_000,
        wall_time="2h 15m 30s",
        round=1,
        best_eval_reward="274.8",
        final_eval_reward="250.3",
        human_score="30.5",
        human_normalized_score="94.87%",
        published_ppo_score="274.8",
        best_model_path="outputs/breakout/best_model.zip",
        final_model_path="outputs/breakout/final_model.zip",
        tensorboard_log_dir="outputs/breakout/tb_logs",
        generated_at=datetime.now().strftime("%Y-%m-%d %H:%M"),
    )
    ipy_display(Markdown(demo_card))

---
## 5. Leaderboard

The leaderboard shows the **best result per game**, ranked by
**human-normalized score** (HNS).  HNS is computed as:

$$
\text{HNS} = \frac{\text{agent\_score} - \text{random\_score}}{\text{human\_score} - \text{random\_score}}
$$

An HNS of 100% means the agent matches human performance.  Values above 100%
mean the agent is superhuman on that game.

In [ ]:
from golds.results.baselines import BASELINES, human_normalized_score as compute_hns

try:
    store = ResultStore(path=RESULTS_PATH)
    leaderboard = store.get_leaderboard()

    if not leaderboard:
        print("No results to build a leaderboard. Train some models first!")
        print("  uv run golds train run --config configs/games/breakout.yaml")
    else:
        rows = []
        for r in leaderboard:
            # Compute HNS from baselines if not stored on the result
            hns = r.human_normalized_score
            if hns is None and r.best_eval_reward is not None:
                hns = compute_hns(r.game_id, r.best_eval_reward)

            baseline = BASELINES.get(r.game_id)
            rows.append({
                "Rank": 0,  # filled below
                "Game": r.game_id,
                "Best Reward": r.best_eval_reward,
                "Human Score": baseline.human if baseline else r.human_score,
                "Random Score": baseline.random if baseline else None,
                "Published PPO": baseline.ppo if baseline else r.published_ppo_score,
                "HNS": hns,
                "Timesteps": r.total_timesteps_completed,
            })

        # Sort by HNS descending (None values go to the bottom)
        rows.sort(key=lambda x: x["HNS"] if x["HNS"] is not None else -999, reverse=True)
        for i, row in enumerate(rows, 1):
            row["Rank"] = i

        if HAS_PANDAS:
            df_lb = pd.DataFrame(rows)
            # Format columns for display
            format_dict = {
                "Best Reward": lambda x: f"{x:.1f}" if x is not None else "--",
                "Human Score": lambda x: f"{x:.1f}" if x is not None else "--",
                "Random Score": lambda x: f"{x:.1f}" if x is not None else "--",
                "Published PPO": lambda x: f"{x:.1f}" if x is not None else "--",
                "HNS": lambda x: f"{x:.2%}" if x is not None else "--",
                "Timesteps": lambda x: f"{x:,}",
            }
            styled = df_lb.style.format(format_dict).set_caption(
                "GOLDS Leaderboard (sorted by Human-Normalized Score)"
            )
            display(styled)
        else:
            print("GOLDS Leaderboard (sorted by Human-Normalized Score)")
            print("=" * 95)
            header = f"{'#':>3} {'Game':<22} {'Best':>10} {'Human':>10} {'Random':>10} {'PPO':>10} {'HNS':>10} {'Steps':>14}"
            print(header)
            print("-" * 95)
            for r in rows:
                best = f"{r['Best Reward']:.1f}" if r['Best Reward'] is not None else "--"
                human = f"{r['Human Score']:.1f}" if r['Human Score'] is not None else "--"
                rand = f"{r['Random Score']:.1f}" if r['Random Score'] is not None else "--"
                ppo = f"{r['Published PPO']:.1f}" if r['Published PPO'] is not None else "--"
                hns = f"{r['HNS']:.2%}" if r['HNS'] is not None else "--"
                steps = f"{r['Timesteps']:,}"
                print(f"{r['Rank']:>3} {r['Game']:<22} {best:>10} {human:>10} {rand:>10} {ppo:>10} {hns:>10} {steps:>14}")

except Exception as e:
    print(f"Could not build leaderboard: {e}")
    print("\nOnce you have trained results, the leaderboard will show:")
    print("  - Best reward per game")
    print("  - Comparison to human and published PPO scores")
    print("  - Human-normalized score (HNS) ranking")

---
## Next Steps

1. **Train more games** to populate the leaderboard.
2. **Resume training** to push HNS closer to (or above) 100%.
3. **Export cards** -- copy the Markdown output into your project docs or paper appendix.
4. **Add training curves** -- extend the model card template with inline matplotlib plots from TensorBoard logs.